# Notebook 18: Parallelism Strategies

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebook 02 (Transformer Architecture), Notebook 11 (Continuous Batching)

---

## Background: When One GPU Isn't Enough

An H100 has 80 GB of HBM memory. A LLaMA-3.1-70B model in BF16 requires 140 GB just for weights. You need at least two H100s just to load the model — and that's before you add any KV cache or activations.

Splitting a model across multiple GPUs is called **model parallelism**, and it's a non-trivial engineering challenge. Unlike data parallelism (where you run independent copies of the same model on different batches), model parallelism requires GPUs to communicate at every forward pass.

The LLM community developed three distinct strategies, each with different tradeoffs:

### Tensor Parallelism (TP)

The dominant approach for inference. Introduced for LLMs by the Megatron-LM paper (Shoeybi et al. 2019, NVIDIA). Each linear layer's weight matrix is **split column-wise or row-wise** across GPUs. Every forward pass requires an **AllReduce** communication to combine partial results.

**Example**: A `d_model=8192` linear layer has weight matrix [8192, 8192]. With TP=8 (8 GPUs), each GPU holds [8192, 1024] (column split) or [1024, 8192] (row split). The partial outputs are summed across GPUs with AllReduce.

**Pros**: Very low latency (communication happens mid-layer). Works well for large batches.  
**Cons**: Requires high-bandwidth NVLink. Communication overhead dominates at batch_size=1.

### Pipeline Parallelism (PP)

Split the model **by layers**. GPU 0 runs layers 0–7, GPU 1 runs layers 8–15, etc. Activations are passed sequentially between GPUs. To hide the sequential latency, micro-batches are pipelined.

**Pros**: Communication is only between adjacent stages (point-to-point, not AllReduce). Works over slower interconnects.  
**Cons**: Pipeline bubbles — GPUs idle at the start/end of a batch.

### Sequence Parallelism (SP)

Split computation along the **sequence dimension** rather than model dimension. Each GPU processes a different chunk of the sequence. Used for very long contexts where attention computation itself is the bottleneck. Typically combined with TP.

---

## What You'll Build

1. **Tensor Parallelism** — column and row splits, correctness verification
2. **Pipeline Parallelism** — stage scheduling, bubble analysis
3. **Communication overhead model** — when does parallelism hurt vs help?
4. **Optimal configuration guide** — decision framework for common model/hardware pairs

In [ ]:
!pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

torch.manual_seed(42)
print("Setup complete.")

## Part 1: Tensor Parallelism — Column and Row Splits

The key insight of TP: split each linear layer so that each GPU does less work,
then AllReduce to combine. The MLP of a transformer has two linear layers:
- Layer 1 (W1): [d_model, d_ff] — split along d_ff dimension (column split)
- Layer 2 (W2): [d_ff, d_model] — split along d_ff dimension (row split)

With TP=N: each GPU holds [d_model, d_ff/N] and [d_ff/N, d_model].
One AllReduce after layer 2 combines partial outputs.

In [ ]:
def simulate_tensor_parallel_mlp(batch, seq, d_model, d_ff, tp_size):
    """Verify TP-sharded MLP produces the same result as single-GPU."""
    # Reference weights (single GPU)
    W1 = torch.randn(d_model, d_ff) * 0.02
    W2 = torch.randn(d_ff, d_model) * 0.02
    x = torch.randn(batch, seq, d_model)
    ref_out = torch.relu(x @ W1) @ W2

    # TP-sharded version
    col_size = d_ff // tp_size
    partial_h = []
    for rank in range(tp_size):
        W1_shard = W1[:, rank*col_size:(rank+1)*col_size]
        partial_h.append(torch.relu(x @ W1_shard))

    row_size = d_ff // tp_size
    partial_out = []
    for rank in range(tp_size):
        W2_shard = W2[rank*row_size:(rank+1)*row_size, :]
        partial_out.append(partial_h[rank] @ W2_shard)

    # AllReduce: sum partial outputs
    tp_out = sum(partial_out)

    max_diff = (tp_out - ref_out).abs().max().item()
    return {
        'max_diff': max_diff,
        'correct': max_diff < 1e-4,
        'params_per_gpu': (d_model * d_ff + d_ff * d_model) // tp_size,
        'allreduce_bytes': batch * seq * d_model * 2,
    }

# Test
result = simulate_tensor_parallel_mlp(2, 64, 4096, 16384, tp_size=8)
print(f"TP=8 correctness: max_diff={result['max_diff']:.2e}  {'✅' if result['correct'] else '❌'}")
print(f"Params per GPU (TP=8): {result['params_per_gpu']:,}  (1/8 of total)")
print(f"AllReduce bytes per forward pass: {result['allreduce_bytes']/1e6:.1f} MB")

print("\nScaling:")
print(f"{'TP Size':>10} {'Params/GPU (M)':>16} {'AllReduce (MB)':>16}")
print("-" * 45)
for tp in [1, 2, 4, 8]:
    r = simulate_tensor_parallel_mlp(2, 1, 8192, 32768, tp)
    print(f"{tp:>10} {r['params_per_gpu']/1e6:>16.1f} {r['allreduce_bytes']/1e6:>16.2f}")

## Part 2: Tensor Parallel Attention

Attention can also be parallelized: split the Q, K, V projection across heads.
With TP=N, each GPU handles n_heads/N attention heads. Output projection uses row-parallel split.

In [ ]:
def simulate_tensor_parallel_attention(batch, seq, d_model, n_heads, tp_size):
    """Verify TP-sharded multi-head attention produces same result as single-GPU."""
    head_dim = d_model // n_heads
    heads_per_gpu = n_heads // tp_size

    # Reference
    Wq = torch.randn(d_model, d_model) * 0.02
    Wk = torch.randn(d_model, d_model) * 0.02
    Wv = torch.randn(d_model, d_model) * 0.02
    Wo = torch.randn(d_model, d_model) * 0.02
    x = torch.randn(batch, seq, d_model)

    # Reference full attention
    Q = (x @ Wq).view(batch, seq, n_heads, head_dim).transpose(1,2)
    K = (x @ Wk).view(batch, seq, n_heads, head_dim).transpose(1,2)
    V = (x @ Wv).view(batch, seq, n_heads, head_dim).transpose(1,2)
    attn = torch.softmax(Q @ K.transpose(-2,-1) / head_dim**0.5, dim=-1)
    ref_out = (attn @ V).transpose(1,2).contiguous().view(batch, seq, d_model) @ Wo

    # TP-sharded attention: each GPU processes heads_per_gpu heads
    partial_outs = []
    col_per_gpu = d_model // tp_size  # n_heads * head_dim / tp_size
    for rank in range(tp_size):
        start = rank * col_per_gpu
        end = (rank + 1) * col_per_gpu

        Wq_s = Wq[:, start:end]
        Wk_s = Wk[:, start:end]
        Wv_s = Wv[:, start:end]
        Wo_s = Wo[start:end, :]  # row slice

        Q_s = (x @ Wq_s).view(batch, seq, heads_per_gpu, head_dim).transpose(1,2)
        K_s = (x @ Wk_s).view(batch, seq, heads_per_gpu, head_dim).transpose(1,2)
        V_s = (x @ Wv_s).view(batch, seq, heads_per_gpu, head_dim).transpose(1,2)
        attn_s = torch.softmax(Q_s @ K_s.transpose(-2,-1) / head_dim**0.5, dim=-1)
        out_s = (attn_s @ V_s).transpose(1,2).contiguous().view(batch, seq, col_per_gpu)
        partial_outs.append(out_s @ Wo_s)

    tp_out = sum(partial_outs)  # AllReduce
    max_diff = (tp_out - ref_out).abs().max().item()
    return {'max_diff': max_diff, 'correct': max_diff < 1e-4}

r = simulate_tensor_parallel_attention(2, 32, 4096, 32, tp_size=4)
print(f"TP=4 attention correctness: max_diff={r['max_diff']:.2e}  {'✅' if r['correct'] else '❌'}")

## Part 3: Pipeline Parallelism — Bubbles

In [ ]:
def pipeline_bubble_fraction(n_stages, n_micro_batches):
    """
    Bubble fraction = (n_stages - 1) / (n_micro_batches + n_stages - 1).
    Rule of thumb: n_micro_batches >> n_stages to amortize bubbles.
    """
    return (n_stages - 1) / (n_micro_batches + n_stages - 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bubble schedule visualization
ax = axes[0]
n_stages, n_mb = 4, 8
colors = plt.cm.tab10(np.linspace(0, 0.8, n_mb))
for stage in range(n_stages):
    for mb in range(n_mb):
        ax.broken_barh([(stage + mb, 1)], (stage * 2, 1.8),
                       facecolors=[colors[mb]], edgecolors='white', linewidth=0.5)
        ax.text(stage + mb + 0.5, stage * 2 + 0.9, f'F{mb+1}',
                ha='center', va='center', fontsize=7, color='white', fontweight='bold')

bubble = pipeline_bubble_fraction(n_stages, n_mb)
ax.set_yticks([s*2+0.9 for s in range(n_stages)])
ax.set_yticklabels([f'GPU {s}' for s in range(n_stages)])
ax.set_xlabel('Time Step')
ax.set_title(f'Pipeline Schedule: {n_stages} stages, {n_mb} micro-batches\nBubble: {bubble:.1%}  Efficiency: {1-bubble:.1%}')
ax.grid(alpha=0.3, axis='x')

# Right: bubble fraction heatmap
ax2 = axes[1]
stages_range = [2, 4, 8, 16]
mb_range = [2, 4, 8, 16, 32, 64]
grid = np.array([[pipeline_bubble_fraction(ns, mb) for mb in mb_range] for ns in stages_range])
im = ax2.imshow(grid, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.5)
ax2.set_xticks(range(len(mb_range)))
ax2.set_xticklabels(mb_range)
ax2.set_yticks(range(len(stages_range)))
ax2.set_yticklabels(stages_range)
ax2.set_xlabel('Micro-batches')
ax2.set_ylabel('Pipeline Stages')
ax2.set_title('Bubble Fraction Heatmap')
for i in range(len(stages_range)):
    for j in range(len(mb_range)):
        ax2.text(j, i, f'{grid[i,j]:.0%}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax2)

plt.tight_layout()
plt.savefig('pipeline_parallelism.png', dpi=120, bbox_inches='tight')
plt.show()

print("Rule: at n_micro_batches = 4 * n_stages, bubble fraction < 20%")

## Part 4: Communication Overhead Model

In [ ]:
@dataclass
class HardwareConfig:
    name: str
    intra_bw_gbs: float   # NVLink bandwidth
    inter_bw_gbs: float   # InfiniBand bandwidth
    gpu_mem_gb: float
    base_decode_ms: float  # single-GPU decode for 7B

hw_h100 = HardwareConfig('H100 SXM', 900, 400, 80, 3.5)

def decode_latency_ms(model_gb, tp, pp, hw, batch=1, seq=1, d_model=8192, n_layers=80):
    """Estimate decode latency with TP + PP parallelism."""
    # Compute: load model_gb/tp/pp weights from memory
    compute = hw.base_decode_ms * (7 / (model_gb / 2)) / (tp * pp)

    # TP AllReduce per layer (each GPU emits d_model*2 bytes)
    if tp > 1:
        allreduce_vol = batch * seq * d_model * 2  # bytes
        tp_comm = 2 * (tp-1)/tp * allreduce_vol / (hw.intra_bw_gbs * 1e9) * 1000
        tp_comm *= (n_layers / pp)
    else:
        tp_comm = 0

    # PP point-to-point activation transfer
    if pp > 1:
        act_vol = batch * seq * d_model * 2
        pp_comm = act_vol / (hw.inter_bw_gbs * 1e9) * 1000 * (pp - 1)
    else:
        pp_comm = 0

    total = compute + tp_comm + pp_comm
    return compute, tp_comm, pp_comm, total

print("Decode latency estimates (70B, BF16) on H100 cluster:")
print(f"{'Config':<14} {'GPUs':>5} {'Compute':>10} {'TP Comm':>10} {'PP Comm':>10} {'Total':>10}")
print("-" * 60)
model_gb_70b = 140.0  # 70B * 2 bytes

for tp, pp in [(1,4),(2,2),(4,1),(8,1),(4,2),(8,2)]:
    gpus = tp * pp
    gpu_mem_needed = model_gb_70b / (tp * pp)
    if gpu_mem_needed > hw_h100.gpu_mem_gb:
        print(f"  TP={tp} PP={pp:<4}  {gpus:>5}  OOM (needs {gpu_mem_needed:.0f}GB/GPU)")
        continue
    c, tc, pc, total = decode_latency_ms(model_gb_70b, tp, pp, hw_h100)
    print(f"  TP={tp} PP={pp:<4}  {gpus:>5}  {c:>9.1f}ms {tc:>9.2f}ms {pc:>9.2f}ms {total:>9.1f}ms")

## Part 5: Configuration Decision Guide

In [ ]:
print('''
Multi-GPU Parallelism Decision Guide
=====================================

1. Fits on one GPU? → No parallelism needed. Single GPU is simplest.

2. Have NVLink (datacenter)?
   → Yes: Use Tensor Parallelism within node (TP=2,4,8)
   → No (PCIe/consumer): Avoid TP. Use offload or PP.

3. Exceeds node memory even with TP=8?
   → Add Pipeline Parallelism across nodes (PP=2,4,...)

4. Very long context (>32K)?
   → Add Sequence Parallelism on top of TP

Common Configurations (H100 80GB):
  Model      |  Config  | GPUs | Notes
  -----------|----------|------|---------------------------
  7B  BF16   |  TP=1    |  1   | Fits easily
  13B BF16   |  TP=1    |  1   | 26GB, fits on 80GB H100
  34B BF16   |  TP=2    |  2   | 68GB per GPU without TP
  70B BF16   |  TP=4    |  4   | Standard config
  70B BF16   |  TP=8    |  8   | Lower latency, more cost
  405B BF16  |  TP=8    |  8   | 101GB/GPU — tight
  405B BF16  |TP=8,PP=2 |  16  | Comfortable fit

vLLM CLI:
  vllm serve model --tensor-parallel-size 4
  vllm serve model --tensor-parallel-size 8 --pipeline-parallel-size 2
''')

## Summary

Multi-GPU parallelism is essential for large model inference. The right strategy depends on interconnect bandwidth and model size.

### Key Takeaways

**Tensor Parallelism (TP)**  
Split weight matrices across GPUs. Requires NVLink for low-latency AllReduce. The standard approach for datacenter inference: TP=4 for 70B, TP=8 for 140B+. Zero quality impact — it's numerically identical to single-GPU.

**Pipeline Parallelism (PP)**  
Split layers across GPU groups. Works over slower interconnects. Adds latency from pipeline bubbles — use many micro-batches to amortize. Typically combined with TP for very large models.

**Communication Is the Bottleneck**  
At batch_size=1 (interactive serving), AllReduce for TP can dominate compute. This is why TP over PCIe is often counterproductive — the communication kills the benefit of parallelism.

### What's Next (Notebook 19: LLM Routing & Cost Optimization)

With a deep understanding of inference costs, notebook 19 covers intelligent request routing: directing simple queries to cheap small models and complex queries to capable large models. The goal is maintaining quality while dramatically reducing cost — the `dispatch` tool from the project backlog.